In [25]:
import pandas as pd

In [71]:
path_datasets = "datasets/"
path_csvs = path_datasets + "csvs/"

# 9n

In [27]:
path_9n = path_datasets + "9n.txt"
df = pd.read_csv(path_9n, sep= ' ')
df.to_csv(path_csvs + "9n.csv", index=False, encoding='utf-8')

# NAT

In [28]:
path = path_datasets + "nat.txt"
df = pd.read_csv(path, sep= ' ')
df.to_csv(path_csvs + "nat.csv", index=False, encoding='utf-8')

In [29]:
import time
from datetime import datetime
def calc_date_hourly(date):
    # Formato del datetime
    date_format = "%a %b %d %H:%M:%S %z %Y"

    # Parsear la cadena en un objeto datetime
    dt = datetime.strptime(date, date_format)

    # Convertir a tiempo Unix
    unix_time = int(dt.timestamp())

    # Caclcular la "hora unix"
    unix_hour = int(unix_time/3600)

    return unix_hour

# HEBDO

In [30]:
def word2vec(word):
    from collections import Counter
    from math import sqrt

    # count the characters in word
    cw = Counter(word)
    print(cw)
    # precomputes a set of the different characters
    sw = set(cw)
    print(sw)
    # precomputes the "length" of the word vector
    lw = sqrt(sum(c*c for c in cw.values()))
    print(lw)
    # return a tuple
    return cw, sw, lw

def cosdis(v1, v2):
    # which characters are common to the two words?
    common = v1[1].intersection(v2[1])
    # by definition of cosine distance we have
    return sum(v1[0][ch]*v2[0][ch] for ch in common)/v1[2]/v2[2]

In [31]:
cosdis(word2vec("hola me llamo manuel suarez"), word2vec("hello me llmao manuel suarez"))


Counter({'l': 4, 'a': 4, ' ': 4, 'm': 3, 'e': 3, 'o': 2, 'u': 2, 'h': 1, 'n': 1, 's': 1, 'r': 1, 'z': 1})
{'h', 'z', 'l', 'e', 'm', 'u', 'n', 'r', 'a', ' ', 'o', 's'}
8.888194417315589
Counter({'l': 5, 'e': 4, ' ': 4, 'm': 3, 'a': 3, 'o': 2, 'u': 2, 'h': 1, 'n': 1, 's': 1, 'r': 1, 'z': 1})
{'h', 'z', 'l', 'e', 'm', 'u', 'n', 'r', 'a', ' ', 'o', 's'}
9.38083151964686


0.9834651404058792

In [32]:
cosdis(word2vec("hola"), word2vec("cola"))

Counter({'h': 1, 'o': 1, 'l': 1, 'a': 1})
{'h', 'o', 'l', 'a'}
2.0
Counter({'c': 1, 'o': 1, 'l': 1, 'a': 1})
{'c', 'o', 'l', 'a'}
2.0


0.75

In [33]:
from unidecode import unidecode
import re 

def clean_hashtags(hstg):
    hstg = str(unidecode(hstg))
    hstg = hstg.lower()
    hstg = re.sub(r'[^a-zA-Z0-9]', '', hstg)
    if len(hstg) == 0:
        return ''
    if hstg[-1] == 's':
        hstg = hstg[:-1]
    return hstg

In [34]:
# Cargo la información de cada tweet en un array de diccionarios
arr_dicts =[]
path_heb = path_datasets + "mas_data/ch/protest_tweets.France.psv"
with open(path_heb, "r") as f:
    line = f.readline()
    splitted = line[:-1].split('|')
    if line!= "":
        splitted = line[:-1].split('|')
        dict_heb = {}
        dict_heb["date"] = splitted[0]
        date_hourly = calc_date_hourly(splitted[0])
        dict_heb["date_formatted"] = date_hourly
        dict_heb["screenname"] = splitted[1]
        dict_heb["user_id"] = splitted[2]
        dict_heb["lat"] = splitted[3]
        dict_heb["long"] = splitted[4]
        dict_heb["hashtags"] = [clean_hashtags(hst) for hst in splitted[5:]]
        arr_dicts.append(dict_heb)
    while line:
        line = f.readline()
        if line!= "":
            splitted = line[:-1].split('|')
            dict_heb = {}
            dict_heb["date"] = splitted[0] 
            date_hourly = calc_date_hourly(splitted[0])
            dict_heb["date_formatted"] = date_hourly
            dict_heb["screenname"] = splitted[1]
            dict_heb["user_id"] = splitted[2]
            dict_heb["lat"] = splitted[3]
            dict_heb["long"] = splitted[4]
            dict_heb["hashtags"] = [clean_hashtags(hst) for hst in splitted[5:]]
            arr_dicts.append(dict_heb)

In [35]:
# Creo el diccionario que asigna un id único a cada hashtag y lo escribo
hstg_id_counter = 1000
dict_hstg = {}
for dict in arr_dicts:
    hstgs = dict["hashtags"]
    for hstg in hstgs:
        if hstg not in dict_hstg.keys():
            dict_hstg[hstg] = hstg_id_counter
            hstg_id_counter +=  1


In [36]:
len(dict_hstg.keys())

37598

In [37]:
arr_new_keys = []
for cadena in dict_hstg.keys():
    nueva_cadena = clean_hashtags(cadena)
    arr_new_keys.append(nueva_cadena)
    if nueva_cadena != cadena:  # Compara si hubo algún cambio
        print(f"Cambiada: Original: '{cadena}' -> Modificada: '{nueva_cadena}'")

Cambiada: Original: 'madnes' -> Modificada: 'madne'
Cambiada: Original: 'bestforbusines' -> Modificada: 'bestforbusine'
Cambiada: Original: 'badas' -> Modificada: 'bada'
Cambiada: Original: 'shameles' -> Modificada: 'shamele'
Cambiada: Original: 'colloqueiris' -> Modificada: 'colloqueiri'
Cambiada: Original: 'stres' -> Modificada: 'stre'
Cambiada: Original: 'heartofglas' -> Modificada: 'heartofgla'
Cambiada: Original: 'chelmietmosspres' -> Modificada: 'chelmietmosspre'
Cambiada: Original: 'ingres' -> Modificada: 'ingre'
Cambiada: Original: 'weis' -> Modificada: 'wei'
Cambiada: Original: 'bos' -> Modificada: 'bo'
Cambiada: Original: 'tos' -> Modificada: 'to'
Cambiada: Original: 'striveforgreatnes' -> Modificada: 'striveforgreatne'
Cambiada: Original: '243bos' -> Modificada: '243bo'
Cambiada: Original: 'mariobros' -> Modificada: 'mariobro'
Cambiada: Original: 'lonelines' -> Modificada: 'loneline'
Cambiada: Original: 'mandjackdus' -> Modificada: 'mandjackdu'
Cambiada: Original: 'captainja

In [38]:

with open("datasets/dicts_hashtags/dict_hashtags_ch.csv", "w") as f:
    f.write("Hashtag,Id\n")
    for hstg in dict_hstg.keys():
        f.write(str(hstg) + ',' + str(dict_hstg[hstg]) + '\n')

In [39]:
# Escribo el csv con el formato de 9n y nat en la carpeta csvs
dict_tweets = {}
# key user+hstg+hour

for tw in arr_dicts:
    for hstg in tw["hashtags"]:
        if str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg]) not in dict_tweets.keys():
            dict_tweets[str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg])] = 1
        else:
            dict_tweets[str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg])] += 1

In [40]:

with open( path_csvs + "hb.csv", "w") as f:
    f.write("user,hashtag,hour,weight\n")
    for k in dict_tweets.keys():
        keys = k.split('|')
        f.write(keys[0] + ',' + keys[2] + ',' + keys[1] + ',' + str(dict_tweets[k]) + '\n') 


# Womanday

In [41]:
# Cargo la información de cada tweet en un array de diccionarios
arr_dicts_wd =[]
path_wd = path_datasets + "mas_data/wd/protest_tweets.2018.United States.psv"
with open(path_wd, "r") as f:
    line = f.readline()
    splitted = line[:-1].split('|')
    if line!= "":
        splitted = line[:-1].split('|')
        dict_wd = {}
        dict_wd["date"] = splitted[0]
        date_hourly = calc_date_hourly(splitted[0])
        dict_wd["date_formatted"] = date_hourly
        dict_wd["screenname"] = splitted[1]
        dict_wd["user_id"] = splitted[2]
        dict_wd["lat"] = splitted[3]
        dict_wd["long"] = splitted[4]
        dict_wd["hashtags"] = [clean_hashtags(hst) for hst in splitted[5:]]
        arr_dicts_wd.append(dict_wd)
    while line:
        line = f.readline()
        if line!= "":
            splitted = line[:-1].split('|')
            dict_wd = {}
            dict_wd["date"] = splitted[0]
            date_hourly = calc_date_hourly(splitted[0])
            dict_wd["date_formatted"] = date_hourly
            dict_wd["screenname"] = splitted[1]
            dict_wd["user_id"] = splitted[2]
            dict_wd["lat"] = splitted[3]
            dict_wd["long"] = splitted[4]
            dict_wd["hashtags"] = [clean_hashtags(hst) for hst in splitted[5:]]
            arr_dicts_wd.append(dict_wd)

In [42]:
# Creo el diccionario que asigna un id único a cada hashtag y lo escribo
hstg_id_counter = 1000
dict_hstg = {}
for dict in arr_dicts_wd:
    hstgs = dict["hashtags"]
    for hstg in hstgs:
        if hstg not in dict_hstg.keys():
            dict_hstg[hstg] = hstg_id_counter
            hstg_id_counter +=  1

with open("datasets/dicts_hashtags/dict_hashtags_wd.csv", "w") as f:
    f.write("Hashtag,Id\n")
    for hstg in dict_hstg.keys():
        f.write(str(hstg) + ',' + str(dict_hstg[hstg]) + '\n')

In [43]:
# Escribo el csv con el formato de 9n y nat en la carpeta csvs
dict_tweets = {}
# key user+hstg+hour

for tw in arr_dicts_wd:
    for hstg in tw["hashtags"]:
        if str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg]) not in dict_tweets.keys():
            dict_tweets[str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg])] = 1
        else:
            dict_tweets[str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg])] += 1

In [44]:

with open( path_csvs + "wd.csv", "w") as f:
    f.write("user,hashtag,hour,weight\n")
    for k in dict_tweets.keys():
        keys = k.split('|')
        f.write(keys[0] + ',' + keys[2] + ',' + keys[1] + ',' + str(dict_tweets[k]) + '\n') 


# Más Datasets

## USELECTION2020 (Trump)

In [45]:
def calc_date_hourly2(date):
    """
    Convierte una cadena con formato '2020-01-03 14:26:06'
    a un entero que representa el tiempo Unix.

    :param date_string: Fecha en formato string.
    :return: Entero que representa el tiempo Unix.
    """
    # Formato del datetime
    date_format = "%Y-%m-%d %H:%M:%S"

    # Parsear la cadena en un objeto datetime
    dt = datetime.strptime(date, date_format)

    # Convertir a tiempo Unix
    unix_time = int(dt.timestamp())

    unix_time = int(unix_time/3600)

    return unix_time

In [46]:
import re

def extract_hashtags(tweet):
    """Extract hashtags from a tweet."""
    return re.findall(r'#\w+', str(tweet))

def generate_unique_ids(items):
    """Generate unique IDs for a list of items."""
    unique_items = list(set(items))
    return {item: idx for idx, item in enumerate(unique_items, start=1)}

def process_tweets(input_csv, output_csv):
    # Load the input CSV
    df = pd.read_csv(input_csv)
    df.dropna(how='all', inplace=True)
    # Extract all hashtags from tweets
    df['hashtags'] = df['text'].apply(extract_hashtags)

    # Flatten list of hashtags and generate unique IDs for users and hashtags
    all_users = df['username'].unique()
    all_hashtags = [hashtag for hashtags in df['hashtags'] for hashtag in hashtags]

    user_id_map = generate_unique_ids(all_users)
    hashtag_id_map = generate_unique_ids(all_hashtags)


    # Create the new DataFrame with user_id, hashtag_id, and datetime
    rows = []
    for _, row in df.iterrows():
        user_id = user_id_map[row['username']]
        for hashtag in row['hashtags']:
            hashtag_id = hashtag_id_map[hashtag]
            #Convert from datatime to hou windows
            date_hourly = calc_date_hourly2(row['tweetcreated'])
            rows.append({'user': user_id, 'hashtag': hashtag_id, 'hour': date_hourly})

    output_df = pd.DataFrame(rows)

    # Add weight column to count occurrences of each triplet
    weighted_df = output_df.groupby(['user', 'hashtag', 'hour']).size().reset_index(name='weight')

    # Save the weighted DataFrame to a new CSV file
    weighted_df.to_csv(output_csv, index=False)

# Example usage
process_tweets('datasets/mas_data/USElectionDayTrump2020/247500_totaloutput_9parts.csv', path_csvs + 'usa.csv')


/tmp/ipykernel_3934953/1812758663.py:14: DtypeWarning: Columns (0,1,2,5,6,7,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


# RM - Lvpl

In [1]:
path_fut = "datasets/mas_data/ChampionsMadridLvp2018/TweetsChampions.json"

In [54]:
import pandas as pd
from tqdm import tqdm
import ast

df = pd.read_csv("datasets/mas_data/ChampionsMadridLvp2018/fut_unfiltered.csv")
# df = pd.read_json(path_fut, lines=True)
#df.to_csv("datasets/csvs/fut_unfiltered.csv", index=False)
df = df[['text', 'user', 'timestamp_ms']].dropna()

/tmp/ipykernel_20499/4091252609.py:5: DtypeWarning: Columns (13,14,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("datasets/mas_data/ChampionsMadridLvp2018/fut_unfiltered.csv")


In [55]:
def process_user(x):
    try:
        if isinstance(x, str):
            user_dict = ast.literal_eval(x) if isinstance(x, str) else x
            return user_dict["id"]
        else:
            # Si no es un string (posiblemente un número), devuelve el valor directamente
            return x
    except Exception as e:
        print(f"Error en Valor {x}")
        return None

df['user'] = df['user'].apply(process_user)
df.rename(columns={"user": "user_id"}, inplace=True, errors="raise")

In [57]:
from unidecode import unidecode
import re

def clean_hashtags(hstg):
    hstg = str(unidecode(hstg))
    hstg = hstg.lower()
    hstg = re.sub(r'[^a-zA-Z0-9]', '', hstg)
    if len(hstg) == 0:
        return ''
    if hstg[-1] == 's':
        hstg = hstg[:-1]
    return hstg

def extract_hashtags(tweet):
    """Extract hashtags from a tweet."""
    hstgs = re.findall(r'#\w+', str(tweet))
    hstgs_arr = []
    for hstg in hstgs:
        hstgs_arr.append(clean_hashtags(hstg))
    return hstgs_arr

df["text"] = df["text"].apply(extract_hashtags)
df.rename(columns={"text": "hashtags"}, inplace=True, errors="raise")

In [58]:
def calc_date_hourlyfut(date):
    # Formato del datetime
    fecha = pd.to_datetime(date)

    # Convertir a tiempo Unix
    unix_time = int(fecha.timestamp())

    # Caclcular la "hora unix"
    unix_hour = int(unix_time/3600)

    return unix_hour
df["timestamp_ms"] = df["timestamp_ms"].apply(calc_date_hourlyfut)
df.rename(columns={"timestamp_ms": "date_formatted"}, inplace=True, errors="raise")

In [68]:
arr_dicts_fut = df.to_dict(orient="records")

In [60]:
# Creo el diccionario que asigna un id único a cada hashtag y lo escribo
hstg_id_counter = 1000
dict_hstg = {}
for dict in arr_dicts_fut:
    hstgs = dict["hashtags"]
    for hstg in hstgs:
        if hstg not in dict_hstg.keys():
            dict_hstg[hstg] = hstg_id_counter
            hstg_id_counter +=  1

with open("datasets/dicts_hashtags/dict_hashtags_fut.csv", "w") as f:
    f.write("Hashtag,Id\n")
    for hstg in dict_hstg.keys():
        f.write(str(hstg) + ',' + str(dict_hstg[hstg]) + '\n')

In [69]:
arr_dicts_fut

[{'hashtags': ['uclfinal', 'lfc'],
  'user_id': 2846595478,
  'date_formatted': 424261},
 {'hashtags': ['uclfinal', 'sportone'],
  'user_id': 2917613580,
  'date_formatted': 424261},
 {'hashtags': ['innovateyourgame', 'uclfinal'],
  'user_id': 946054253461852160,
  'date_formatted': 424261},
 {'hashtags': [], 'user_id': 962840767, 'date_formatted': 424261},
 {'hashtags': ['uclfinal'],
  'user_id': 902735000445095938,
  'date_formatted': 424261},
 {'hashtags': [], 'user_id': 240672622, 'date_formatted': 424261},
 {'hashtags': ['uclfinal'], 'user_id': 311037764, 'date_formatted': 424261},
 {'hashtags': ['uclfinal'], 'user_id': 277019564, 'date_formatted': 424261},
 {'hashtags': ['uclfinal', 'laysunited'],
  'user_id': 957644286350315521,
  'date_formatted': 424261},
 {'hashtags': ['uclfinal'], 'user_id': 1672941752, 'date_formatted': 424261},
 {'hashtags': ['uclfinal'], 'user_id': 4057128733, 'date_formatted': 424261},
 {'hashtags': ['uclfinal', 'weareliverpool'],
  'user_id': 2485857084

In [72]:
# Escribo el csv con el formato de 9n y nat en la carpeta csvs
dict_tweets = {}
# key user+hstg+hour

for tw in arr_dicts_fut:
    for hstg in tw["hashtags"]:
        if str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg]) not in dict_tweets.keys():
            dict_tweets[str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg])] = 1
        else:
            dict_tweets[str(tw["user_id"]) + '|' + str(tw["date_formatted"]) + '|' + str(dict_hstg[hstg])] += 1


with open(path_csvs + "fut.csv", "w") as f:
    f.write("user,hashtag,hour,weight\n")
    for k in dict_tweets.keys():
        keys = k.split('|')
        f.write(keys[0] + ',' + keys[2] + ',' + keys[1] + ',' + str(dict_tweets[k]) + '\n') 
